# Plot all Sz, SD1, SD2 (TMEV CA1) directions on medial/lateral axis

In [ ]:
import sys
import os
import pandas as pd
notebook_dir = os.getcwd()
sys.path.insert(0, os.path.abspath(os.path.join(notebook_dir, '..')))  # Add the project root directory to the path
import custom_io as cio
import env_reader
import metadata_reader
from directionality_analysis import stdOfUniformAngles
import matplotlib.pyplot as plt
import h5py
import numpy as np
import scipy.io
from math import pi

In [ ]:
save_fig = True

In [ ]:
er = env_reader.read_env()

In [ ]:
mr = metadata_reader.MetadataReader.from_env_dict(er)
#mr.load_metadata_reader()

In [ ]:
df_colorings = mr.colorings_df

In [ ]:
fpath_absolute_angles = cio.open_file("Open excel file with absolute angles per event component (Sz, SD1, SD2)")
print(fpath_absolute_angles)

In [ ]:
df_absolute_angles = pd.read_excel(fpath_absolute_angles)

In [ ]:
df_absolute_angles = df_absolute_angles[df_absolute_angles["win_type"] == "CA1"]

In [ ]:
event_types = ["SD2", "SD1", "Sz"]
dict_event_types_radii = {event.lower(): i+1 for i, event in enumerate(event_types)}  # seizure has largest radius in categorical plot

In [ ]:
# Form vectorial averaging of angles
# 1. Convert angles to cos and sin components
df_absolute_angles['cos'] = np.cos(df_absolute_angles["theta_inj_top"])
df_absolute_angles['sin'] = np.sin(df_absolute_angles["theta_inj_top"])
# 2. Calculate the mean of the cos and sin components
df_grouped = df_absolute_angles.groupby(['mouse_id', 'event_type'])[["cos", "sin"]].mean().reset_index()
df_grouped["theta_inj_top"] = np.arctan2(df_grouped["sin"], df_grouped["cos"])
df_grouped["r_categorical"] = df_grouped["event_type"].map(dict_event_types_radii)
df_grouped["theta_inj_top_deg"] = df_grouped["theta_inj_top"] * 180 / pi

In [ ]:
df_mean_per_event_type = df_grouped.groupby(['event_type'])[["cos", "sin"]].mean().reset_index()
df_mean_per_event_type["theta_inj_top"] = np.arctan2(df_mean_per_event_type["sin"], df_mean_per_event_type["cos"])
df_mean_per_event_type["r_categorical"] = df_mean_per_event_type["event_type"].map({"sz": 3, "sd1": 2, "sd2": 1})
df_mean_per_event_type["theta_inj_top_deg"] = df_mean_per_event_type["theta_inj_top"] * 180 / pi

In [ ]:
fig = plt.figure(figsize=(12,12))
ax = fig.add_subplot(111, projection='polar')

# plot arrows
for i, row in df_grouped.iterrows():
    r = row["r_categorical"]
    theta = row['theta_inj_top']
    ax.annotate('', xy=(theta, r), xytext=(0, 0),
                arrowprops=dict(facecolor=df_colorings[df_colorings["mouse_id"] == row["mouse_id"]]["color"].iloc[0], edgecolor='none', width=2, headwidth=6, alpha=0.6))

# add mean directions per event type
for i, row in df_mean_per_event_type.iterrows():
    r = row["r_categorical"]
    theta = row['theta_inj_top']
    ax.annotate('', xy=(theta, r), xytext=(0, 0),
                arrowprops=dict(facecolor='black', edgecolor='none', width=4, headwidth=8, alpha=1))

# set the radial limits

max_r = len(event_types)-1#qdf['r'].max()
ax.set_ylim(0, max_r + 1)
ax.set_rgrids(np.linspace(0, max_r+1, num=4))
ax.set_yticklabels([""] + event_types)

# when at 2p setup,
# button  to right pressed:
# 1. in FoV, we move to right
# 2. mouse moves forward, FoV moves toward posterior
# -> right side of plot is posterior
# (0 degrees; in preparation of data, we only mirror around the x axis to get medial side always on top; so A/P does not change)
ax.set_xticklabels(['posterior', '', 'medial/injection', '', 'anterior', '', 'lateral', ''], fontsize=20)

#ax.set_xticklabels(event_types) 
# display the plot
if save_fig:
    fig.savefig(os.path.join(er["OUTPUT_FOLDER"], "directions_all_ca1_events.pdf"))
plt.show()

In [ ]:
# surrogate analysis
# for each event type (Sz, SD1, SD2), we generate n random angles and calculate the standard deviation; repeat this n_surrogates times.
# Then, we calculate the standard deviation of the mean angles for each event type.
# We then compare the standard deviation of the mean angles to the standard deviation of the random angles.
n_surrogates = 1000
n_event_types = len(event_types)
df_angles = df_grouped.groupby("event_type")["mouse_id"].count().reset_index()  # get n_angles for event type by df_angles[df_angles["event_type"] == event_type]["mouse_id"].iloc[0]
rng = np.random.default_rng(42)
surrogate_data = np.zeros((n_event_types, n_surrogates))

for i_event, event_type in enumerate(event_types):
    n_angles = df_angles[df_angles["event_type"] == event_type.lower()]["mouse_id"].iloc[0]
    random_stds = np.array(sorted([stdOfUniformAngles(rng, n_angles, True) for i in range(n_surrogates)]))  # sort to easily determine alpha level
    surrogate_data[i_event, :] = random_stds

measured_stds = np.zeros(n_event_types)
for i_event, event_type in enumerate(event_types):
    measured_stds[i_event] = np.std(df_grouped[df_grouped["event_type"] == event_type.lower()]["theta_inj_top_deg"])

# plot results

fig, axs = plt.subplots(1, n_event_types, figsize=(n_event_types*8, 8))
for i_event, event in enumerate(event_types):
    ax = axs[i_event]
    ax.hist(surrogate_data[i_event, :], bins=20, alpha=0.5, color='gray')
    ax.axvline(x=measured_stds[i_event], color='red', linestyle='--', label="mean angle stdev.")
    ax.axvline(x=np.percentile(surrogate_data[i_event, :], 5), color='blue', linestyle='--', label="5% percentile")
    ax.set_title(f"Standard deviation of mean angles for {event}")
    ax.set_xlabel("Standard deviation (°)")
    ax.legend()

if save_fig:
    fig.savefig(os.path.join(er["OUTPUT_FOLDER"], "mean_angles_surrogate_test.pdf"))
plt.show()

In [ ]:
# save data to matlab workspace
matlab_data = {
    "event_types": event_types,
    "surrogate_stds": surrogate_data,
    "measured_stds": measured_stds,
}
matlab_data["event_types"] = [event_type.encode('utf-8') for event_type in matlab_data["event_types"]]
matlab_data["event_types"] = np.array(matlab_data["event_types"], dtype=object)
matlab_data["surrogate_stds"] = np.array(matlab_data["surrogate_stds"], dtype=np.float32)
matlab_data["measured_stds"] = np.array(matlab_data["measured_stds"], dtype=np.float32)
matlab_data["surrogate_stds"] = matlab_data["surrogate_stds"].T  # transpose to have event types as first dimension
# save to matlab file
matlab_file_path = os.path.join(er["OUTPUT_FOLDER"], "mean_angles_surrogate_test.mat")
scipy.io.savemat(matlab_file_path, matlab_data)
print(f"Saved data to {matlab_file_path}")
